# 第3节:BERT微调

本节介绍如何将预训练的BERT模型微调应用于各种下游NLP任务,包括序列级和词元级应用。

## 学习目标

1. 理解BERT微调的基本原理和优势
2. 掌握序列级任务的BERT微调方法(单文本分类、文本对分类)
3. 掌握词元级任务的BERT微调方法(文本标注、问答)
4. 实现BERT在自然语言推理任务上的微调
5. 理解微调期间的参数更新策略

## 3.1 BERT微调概述

### 为什么需要BERT微调?

在前面几节中,我们为自然语言处理应用设计了不同的模型,例如:
- 基于**循环神经网络**(RNN)
- 基于**卷积神经网络**(CNN) 
- 基于**注意力机制**和多层感知机

这些模型在有空间或时间限制的情况下是有帮助的,但是,**为每个自然语言处理任务精心设计一个特定的模型实际上是不可行的**。

### BERT的优势

BERT是一个预训练模型,该模型可以对广泛的自然语言处理任务进行**最少的架构更改**:

1. **改进技术水平**: 在提出时,BERT改进了各种自然语言处理任务的技术水平
2. **参数规模大**: 
   - BERT-Base: 1.1亿个参数
   - BERT-Large: 3.4亿个参数
3. **迁移学习**: 当有足够的计算资源时,我们可以考虑为下游自然语言处理应用微调BERT

### 微调策略

在微调期间:
- **额外层的参数**从零开始学习
- **预训练BERT模型中的所有参数**都是微调的
- 不同应用之间的BERT所需的"最小架构更改"是**额外的全连接层**

In [ ]:
import json
import multiprocessing
import os
import torch
from torch import nn
from d2l import torch as d2l

## 3.2 序列级应用

### 3.2.1 单文本分类

**单文本分类**将单个文本序列作为输入,并输出其分类结果。

#### 应用示例

1. **情感分析**: 判断评论是积极还是消极
2. **语言可接受性**: 判断句子在语法上是否可以接受
   - 可接受: "I should study."
   - 不可接受: "I should studying."

![微调BERT用于单文本分类应用](https://zh.d2l.ai/_images/bert-one-seq.svg)

#### BERT输入表示

BERT输入序列明确地表示单个文本和文本对:
- 特殊分类标记**"&lt;cls&gt;"**用于序列分类
- 特殊分类标记**"&lt;sep&gt;"**标记单个文本的结束或分隔成对文本

#### 模型架构

在单文本分类应用中:
1. 特殊分类标记"&lt;cls&gt;"的BERT表示对整个输入文本序列的信息进行编码
2. 作为输入单个文本的表示,它将被送入到由**全连接层组成的小多层感知机**中
3. 输出所有离散标签值的分布

### 3.2.2 文本对分类或回归

#### 文本对分类

**自然语言推理**(NLI)属于文本对分类:
- 输入: 前提-假设对
- 输出: 蕴涵/矛盾/中性

#### 文本对回归

**语义文本相似度**(Semantic Textual Similarity):
- 输入: 句子对
- 输出: 相似度得分(0-5)

示例:
- ("A plane is taking off.", "An air plane is taking off.") → 5.000
- ("A woman is eating something.", "A woman is eating meat.") → 3.000
- ("A woman is dancing.", "A man is talking.") → 0.000

![文本对分类或回归应用的BERT微调](https://zh.d2l.ai/_images/bert-two-seqs.svg)

#### 模型差异

与单文本分类相比:
- **输入表示**有所不同(需要处理两个文本序列)
- 对于文本对回归任务,可以应用细微的更改:
  - 输出连续的标签值
  - 使用均方损失

## 3.3 词元级应用

### 3.3.1 文本标注

**文本标注**(text tagging)是一个词元级任务,其中每个词元都被分配了一个标签。

#### 应用示例:词性标注

**词性标注**(Part-of-Speech Tagging)为每个单词分配词性标记,根据单词在句子中的作用。

例如,在Penn树库II标注集中:
- 句子: "John Smith's car is new"
- 标注: "NNP NNP POS NN VB JJ"
  - NNP: 名词,专有单数
  - POS: 所有格结尾
  - NN: 名词,单数或质量
  - VB: 动词,基本形式
  - JJ: 形容词

![文本标记应用的BERT微调](https://zh.d2l.ai/_images/bert-tagging.svg)

#### 模型架构

与单文本分类相比,唯一的区别在于:
- 输入文本的**每个词元**的BERT表示被送到相同的额外全连接层中
- 输出词元的标签(例如词性标签)

### 3.3.2 问答

**问答**任务反映阅读理解能力。

#### SQuAD数据集

斯坦福问答数据集(Stanford Question Answering Dataset, SQuAD v1.1):
- 由阅读段落和问题组成
- 每个问题的答案只是段落中的一段文本(文本片段)

示例:
> **段落**: "Some experts report that a mask's efficacy is inconclusive. However, mask makers insist that their products, such as N95 respirator masks, can guard against the virus."
>
> **问题**: "Who say that N95 respirator masks can guard against the virus?"
>
> **答案**: "mask makers"

目标: 在给定问题和段落的情况下预测段落中**文本片段的开始和结束**。

![对问答进行BERT微调](https://zh.d2l.ai/_images/bert-qa.svg)

#### 模型架构

1. **输入**: 将问题和段落分别作为第一个和第二个文本序列
2. **预测开始位置**:
   - 额外的全连接层将位置$i$的词元的BERT表示转换成标量分数$s_i$
   - 所有词元的分数通过softmax转换成概率分布$p_i$
3. **预测结束位置**:
   - 位置$i$的词元由相同的全连接层变换成标量分数$e_i$
   - 参数与用于预测开始位置的参数**无关**
4. **训练目标**: 最大化真实值的开始和结束位置的对数似然
5. **预测**: 计算从位置$i$到位置$j$的有效片段的分数$s_i + e_j$(i ≤ j),输出分数最高的跨度

## 3.4 实战:BERT用于自然语言推理

现在,我们通过微调BERT来重新审视自然语言推理任务。正如之前讨论的那样,自然语言推断是一个序列级别的文本对分类问题,而微调BERT只需要一个额外的基于多层感知机的架构。

![将预训练BERT提供给基于多层感知机的自然语言推断架构](https://zh.d2l.ai/_images/nlp-map-nli-bert.svg)

本节将下载一个预训练好的小版本的BERT,然后对其进行微调,以便在SNLI数据集上进行自然语言推断。

### 加载预训练的BERT

我们已经在WikiText-2数据集上预训练BERT(请注意,原始的BERT模型是在更大的语料库上预训练的)。原始的BERT模型有数以亿计的参数。

在下面,我们提供了两个版本的预训练的BERT:
- **bert.base**: 与原始的BERT基础模型一样大,需要大量的计算资源才能进行微调
- **bert.small**: 一个小版本,以便于演示

In [ ]:
d2l.DATA_HUB['bert.base'] = (d2l.DATA_URL + 'bert.base.torch.zip',
                             '225d66f04cae318b841a13d32af3acc165f253ac')
d2l.DATA_HUB['bert.small'] = (d2l.DATA_URL + 'bert.small.torch.zip',
                              'c72329e68a732bef0452e4b96a1c341c8910f81f')

两个预训练好的BERT模型都包含:
- 一个定义词表的"vocab.json"文件
- 一个预训练参数的"pretrained.params"文件

我们实现了以下`load_pretrained_model`函数来加载预先训练好的BERT参数。

In [ ]:
def load_pretrained_model(pretrained_model, num_hiddens, ffn_num_hiddens,
                          num_heads, num_layers, dropout, max_len, devices):
    data_dir = d2l.download_extract(pretrained_model)
    # 定义空词表以加载预定义词表
    vocab = d2l.Vocab()
    vocab.idx_to_token = json.load(open(os.path.join(data_dir,
        'vocab.json')))
    vocab.token_to_idx = {token: idx for idx, token in enumerate(
        vocab.idx_to_token)}
    bert = d2l.BERTModel(len(vocab), num_hiddens, norm_shape=[256],
                         ffn_num_input=256, ffn_num_hiddens=ffn_num_hiddens,
                         num_heads=4, num_layers=2, dropout=0.2,
                         max_len=max_len, key_size=256, query_size=256,
                         value_size=256, hid_in_features=256,
                         mlm_in_features=256, nsp_in_features=256)
    # 加载预训练BERT参数
    bert.load_state_dict(torch.load(os.path.join(data_dir,
                                                 'pretrained.params')))
    return bert, vocab

为了便于在大多数机器上演示,我们将在本节中加载和微调经过预训练BERT的小版本("bert.small")。在练习中,我们将展示如何微调大得多的"bert.base"以显著提高测试精度。

In [ ]:
devices = d2l.try_all_gpus()
bert, vocab = load_pretrained_model(
    'bert.small', num_hiddens=256, ffn_num_hiddens=512, num_heads=4,
    num_layers=2, dropout=0.1, max_len=512, devices=devices)

### 微调BERT的数据集

对于SNLI数据集的下游任务自然语言推断,我们定义了一个定制的数据集类`SNLIBERTDataset`。

在每个样本中:
- 前提和假设形成一对文本序列
- 被打包成一个BERT输入序列

回想之前的内容,片段索引用于区分BERT输入序列中的前提和假设。利用预定义的BERT输入序列的最大长度(`max_len`),持续移除输入文本对中较长文本的最后一个标记,直到满足`max_len`。

为了加速生成用于微调BERT的SNLI数据集,我们使用4个工作进程并行生成训练或测试样本。

In [ ]:
class SNLIBERTDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, max_len, vocab=None):
        all_premise_hypothesis_tokens = [[
            p_tokens, h_tokens] for p_tokens, h_tokens in zip(
            *[d2l.tokenize([s.lower() for s in sentences])
              for sentences in dataset[:2]])]

        self.labels = torch.tensor(dataset[2])
        self.vocab = vocab
        self.max_len = max_len
        (self.all_token_ids, self.all_segments,
         self.valid_lens) = self._preprocess(all_premise_hypothesis_tokens)
        print('read ' + str(len(self.all_token_ids)) + ' examples')

    def _preprocess(self, all_premise_hypothesis_tokens):
        pool = multiprocessing.Pool(4)  # 使用4个进程
        out = pool.map(self._mp_worker, all_premise_hypothesis_tokens)
        all_token_ids = [
            token_ids for token_ids, segments, valid_len in out]
        all_segments = [segments for token_ids, segments, valid_len in out]
        valid_lens = [valid_len for token_ids, segments, valid_len in out]
        return (torch.tensor(all_token_ids, dtype=torch.long),
                torch.tensor(all_segments, dtype=torch.long),
                torch.tensor(valid_lens))

    def _mp_worker(self, premise_hypothesis_tokens):
        p_tokens, h_tokens = premise_hypothesis_tokens
        self._truncate_pair_of_tokens(p_tokens, h_tokens)
        tokens, segments = d2l.get_tokens_and_segments(p_tokens, h_tokens)
        token_ids = self.vocab[tokens] + [self.vocab['<pad>']] \
                             * (self.max_len - len(tokens))
        segments = segments + [0] * (self.max_len - len(segments))
        valid_len = len(tokens)
        return token_ids, segments, valid_len

    def _truncate_pair_of_tokens(self, p_tokens, h_tokens):
        # 为BERT输入中的'<CLS>'、'<SEP>'和'<SEP>'词元保留位置
        while len(p_tokens) + len(h_tokens) > self.max_len - 3:
            if len(p_tokens) > len(h_tokens):
                p_tokens.pop()
            else:
                h_tokens.pop()

    def __getitem__(self, idx):
        return (self.all_token_ids[idx], self.all_segments[idx],
                self.valid_lens[idx]), self.labels[idx]

    def __len__(self):
        return len(self.all_token_ids)

下载完SNLI数据集后,我们通过实例化`SNLIBERTDataset`类来生成训练和测试样本。这些样本将在自然语言推断的训练和测试期间进行小批量读取。

In [ ]:
# 如果出现显存不足错误,请减少"batch_size"。在原始的BERT模型中,max_len=512
batch_size, max_len, num_workers = 512, 128, d2l.get_dataloader_workers()
data_dir = d2l.download_extract('SNLI')
train_set = SNLIBERTDataset(d2l.read_snli(data_dir, True), max_len, vocab)
test_set = SNLIBERTDataset(d2l.read_snli(data_dir, False), max_len, vocab)
train_iter = torch.utils.data.DataLoader(train_set, batch_size, shuffle=True,
                                   num_workers=num_workers)
test_iter = torch.utils.data.DataLoader(test_set, batch_size,
                                  num_workers=num_workers)

### 微调BERT

用于自然语言推断的微调BERT只需要一个额外的多层感知机,该多层感知机由两个全连接层组成(请参见下面`BERTClassifier`类中的`self.hidden`和`self.output`)。

这个多层感知机将特殊的"&lt;cls&gt;"词元的BERT表示进行了转换,该词元同时编码前提和假设的信息,为自然语言推断的三个输出:蕴涵、矛盾和中性。

In [ ]:
class BERTClassifier(nn.Module):
    def __init__(self, bert):
        super(BERTClassifier, self).__init__()
        self.encoder = bert.encoder
        self.hidden = bert.hidden
        self.output = nn.Linear(256, 3)

    def forward(self, inputs):
        tokens_X, segments_X, valid_lens_x = inputs
        encoded_X = self.encoder(tokens_X, segments_X, valid_lens_x)
        return self.output(self.hidden(encoded_X[:, 0, :]))

在下文中,预训练的BERT模型`bert`被送到用于下游应用的`BERTClassifier`实例`net`中。

在BERT微调的常见实现中:
- 只有额外的多层感知机(`net.output`)的**输出层的参数**将从零开始学习
- 预训练BERT编码器(`net.encoder`)和额外的多层感知机的隐藏层(`net.hidden`)的**所有参数都将进行微调**

In [ ]:
net = BERTClassifier(bert)

回想一下,`MaskLM`类和`NextSentencePred`类在其使用的多层感知机中都有一些参数。这些参数是预训练BERT模型`bert`中参数的一部分,因此是`net`中的参数的一部分。

然而,这些参数仅用于计算预训练过程中的遮蔽语言模型损失和下一句预测损失。这两个损失函数与微调下游应用无关,因此当BERT微调时,`MaskLM`和`NextSentencePred`中采用的多层感知机的参数**不会更新(陈旧的,staled)**。

为了允许具有陈旧梯度的参数,标志`ignore_stale_grad=True`在`step`函数`d2l.train_batch_ch13`中被设置。我们通过该函数使用SNLI的训练集(`train_iter`)和测试集(`test_iter`)对`net`模型进行训练和评估。

由于计算资源有限,训练和测试精度可以进一步提高:我们把对它的讨论留在练习中。

In [ ]:
lr, num_epochs = 1e-4, 5
trainer = torch.optim.Adam(net.parameters(), lr=lr)
loss = nn.CrossEntropyLoss(reduction='none')
d2l.train_ch13(net, train_iter, test_iter, loss, trainer, num_epochs,
    devices)

## 小结

### 序列级应用

BERT只需要最小的架构改变(额外的全连接层):

1. **单文本分类**:
   - 应用: 情感分析、语言可接受性测试
   - 架构: &lt;cls&gt; token → MLP → 类别分布

2. **文本对分类或回归**:
   - 分类: 自然语言推断(蕴涵/矛盾/中性)
   - 回归: 语义文本相似性(0-5分)
   - 架构: [&lt;cls&gt; 文本1 &lt;sep&gt; 文本2 &lt;sep&gt;] → MLP → 输出

### 词元级应用

1. **文本标注**:
   - 应用: 词性标注、命名实体识别
   - 架构: 每个词元的BERT表示 → 全连接层 → 词元标签

2. **问答**:
   - 应用: 阅读理解(SQuAD)
   - 架构: 
     - 开始位置: 词元表示 → FC → 分数$s_i$ → softmax
     - 结束位置: 词元表示 → FC → 分数$e_i$ → softmax
     - 预测: argmax$(s_i + e_j)$ for $i \leq j$

### 微调策略

- 我们可以针对下游应用对预训练的BERT模型进行微调
- 在微调过程中,BERT模型成为下游应用模型的一部分
- **额外层的参数**从零开始学习
- **预训练BERT的所有参数**都进行微调
- 仅与训练前损失相关的参数(MLM、NSP)在微调期间不会更新

### 优势总结

1. **通用性**: 一个模型适用于多种任务
2. **高效性**: 最小的架构修改
3. **效果好**: 在各种NLP任务上取得SOTA结果
4. **迁移学习**: 充分利用预训练知识

## 练习

### 实验类练习

1. 如果您的计算资源允许,请微调一个更大的预训练BERT模型,该模型与原始的BERT基础模型一样大。修改`load_pretrained_model`函数中的参数设置:
   - 将"bert.small"替换为"bert.base"
   - 将`num_hiddens=256`、`ffn_num_hiddens=512`、`num_heads=4`和`num_layers=2`的值分别增加到768、3072、12和12
   - 通过增加微调迭代轮数(可能还会调优其他超参数),你可以获得高于0.86的测试精度吗?

2. 如何根据一对序列的长度比值截断它们?将此对截断方法与`SNLIBERTDataset`类中使用的方法进行比较。它们的利弊是什么?

### 设计类练习

3. 让我们为新闻文章设计一个搜索引擎算法:
   - 当系统接收到查询(例如,"冠状病毒爆发期间的石油行业")时,它应该返回与该查询最相关的新闻文章的排序列表
   - 假设我们有一个巨大的新闻文章池和大量的查询
   - 为了简化问题,假设为每个查询标记了最相关的文章
   - 如何在算法设计中应用负采样和BERT?

4. 我们如何利用BERT来训练语言模型?

5. 我们能在机器翻译中利用BERT吗?提示:考虑encoder-decoder架构。

### 对比分析练习

6. 比较以下三种NLI模型的优缺点:
   - 可分解注意力模型
   - 基于RNN的模型
   - 微调BERT
   
   从以下方面进行分析:
   - 模型复杂度
   - 参数量
   - 训练时间
   - 推理速度
   - 准确率
   - 可解释性